In [8]:
import pennylane as qml
from model_cqcnn import CNN_QNN_CNN_Geister

# モデル設定
config = {
    "dev_type": "lightning.qubit",
    "n_qubits_qnn": 4,
    "embedding_type": "AngleEmbedding",
    "ansatz_type": "RealAmplitudes",
    "exp_or_prob": "exp",
    "feature_map_reps": 1,
    "ansatz_reps": 1,
    "input_channels_cnn": 6,
    "board_size_cnn": 6,
    "cnn_fc_out_features": 4,
    "qnn_fc_out_features": 36
}

# デバイスとモデル構築
dev = qml.device(config["dev_type"], wires=config["n_qubits_qnn"])
model = CNN_QNN_CNN_Geister(
    dev=dev,
    **{k: v for k, v in config.items() if k != "dev_type"}
)

print("✅ モデル構築完了")


✅ モデル構築完了


In [9]:
from utils import visualize_cqcnn_structure_with_clusters

dot = visualize_cqcnn_structure_with_clusters(
    cnn_layers=["Conv2d(6->16)", "ReLU", "Conv2d(16->32)", "ReLU", "Flatten", "Linear(?->4)"],
    qnn_layers=["AngleEmbedding", "RealAmplitudes", "Expval"]
)
dot.render("cqcnn_model_graph", format="png", view=True)


'cqcnn_model_graph.png'

In [7]:
from model_cqcnn import CNN_QNN_CNN_Geister
from geister_game import GeisterGame

game_instance = GeisterGame()

agent_a = CNN_QNN_CNN_Geister(
    "A", game_instance, dev,
    embedding_type=config["embedding_type"],
    ansatz_type=config["ansatz_type"],
    n_qubits_qnn=config["n_qubits_qnn"],
    input_channels_cnn=config["input_channels_cnn"],
    board_size_cnn=config["board_size_cnn"],
    cnn_fc_out_features=config["cnn_fc_out_features"],
    epsilon=0.5, lr=0.0005
)

agent_b = CNN_QNN_CNN_Geister(
    "B", game_instance, dev,
    embedding_type=config["embedding_type"],
    ansatz_type=config["ansatz_type"],
    n_qubits_qnn=config["n_qubits_qnn"],
    input_channels_cnn=config["input_channels_cnn"],
    board_size_cnn=config["board_size_cnn"],
    cnn_fc_out_features=config["cnn_fc_out_features"],
    epsilon=0.5, lr=0.0005
)

print("✅ エージェント構築完了")


TypeError: CNN_QNN_CNN_Geister.__init__() got multiple values for argument 'embedding_type'

In [ ]:
import pennylane as qml
from pennylane import numpy as np

# デバイス定義
n_qubits = 2
dev_A = qml.device("default.qubit", wires=n_qubits)
dev_B = qml.device("default.qubit", wires=n_qubits)

# Agent A の回路
@qml.qnode(dev_A)
def circuit_A():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0,1])
    return qml.state()

# Agent B の回路
@qml.qnode(dev_B)
def circuit_B():
    # 初期状態はここでは空、後で state をロードします
    return qml.state()

# Agent A の状態を取得
state_A = circuit_A()
print("Agent A の状態ベクトル:")
print(state_A)

# Agent B に Agent A の状態をロード
dev_B._apply_state_vector(state_A, wires=range(n_qubits))

# Agent B の状態を確認
state_B = circuit_B()
print("\nAgent B の状態ベクトル（Agent A の状態をロード後）:")
print(state_B)

# Agent B の状態から追加のゲート処理
@qml.qnode(dev_B)
def circuit_B_with_processing():
    qml.RX(np.pi / 4, wires=0)
    return qml.probs(wires=range(n_qubits))

probs_B = circuit_B_with_processing()
print("\nAgent B の後続処理後の確率分布:")
print(probs_B)

Agent A の状態ベクトル:
[0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


AttributeError: DefaultQubit has no attribute '_apply_state_vector'. You may be looking for a property or method present in the legacy device interface. Please consult the DefaultQubit documentation for an updated list of public properties and methods.

In [12]:
import pennylane as qml
from pennylane import numpy as np

# デバイス
n_qubits = 2
dev_A = qml.device("default.qubit", wires=n_qubits)
dev_B = qml.device("default.qubit", wires=n_qubits)

# Agent A の回路
@qml.qnode(dev_A)
def circuit_A(state):
    qml.StatePrep(state, wires=range(n_qubits))
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0,1])
    return qml.state()

# Agent A の状態を取得
state_A = circuit_A(state=np.array([1, 0, 0, 0]))  # 初期状態をセット
print("Agent A の状態ベクトル:")
print(state_A)

# Agent B の回路 (状態をセットしてから後続ゲート)
@qml.qnode(dev_B)
def circuit_B_with_state(state):
    qml.StatePrep(state, wires=range(n_qubits))
    qml.RX(np.pi / 4, wires=0)
    return qml.probs(wires=range(n_qubits))

# Agent A の状態ベクトルを B にセットして処理
probs_B = circuit_B_with_state(state_A)
print("\nAgent B の後続処理後の確率分布:")
print(probs_B)


Agent A の状態ベクトル:
[0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]

Agent B の後続処理後の確率分布:
[0.4267767 0.0732233 0.0732233 0.4267767]
